In [1]:
import poolparty as pp; pp.init()
import time
import tracemalloc
from pathlib import Path

# --- Basic profiling with timing ---
def time_workload(name, func, *args, **kwargs):
    """Time a function and report results."""
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{name}: {len(result)} seqs in {elapsed:.3f}s ({elapsed/len(result)*1000:.2f}ms/seq)")
    return result

# Mutagenize workload
seq = 'ACGT' * 25  # 100bp
pool = pp.mutagenize(seq, num_mutations=2, mode='random')
time_workload("Mutagenize (100bp, 2mut, 1000 seqs)", pool.generate_library, num_seqs=1000)

# Complex DAG workload
mutants = pp.mutagenize(seq, num_mutations=2, mode='random')
barcode = pp.get_kmers(length=8, mode='random')
complex_pool = pp.join([mutants, '----', barcode])
time_workload("Complex DAG (100bp, 1000 seqs)", complex_pool.generate_library, num_seqs=1000)

# --- Memory profiling with tracemalloc ---
print("\n--- Memory Profiling ---")
tracemalloc.start()
pool = pp.mutagenize('ACGT' * 50, num_mutations=2, mode='random')
df = pool.generate_library(num_seqs=5000)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Mutagenize (200bp, 5000 seqs): current={current/1e6:.2f}MB, peak={peak/1e6:.2f}MB")

# --- Sequential mode (combinatorial) ---
print("\n--- Sequential Mode (Enumerate All States) ---")
start = time.perf_counter()
pool = pp.mutagenize('ACGT' * 5, num_mutations=1, mode='sequential')  # 20bp, 1 mut = 60 states
df = pool.generate_library(num_cycles=1)
elapsed = time.perf_counter() - start
print(f"Sequential mutagenize (20bp, 1mut): {len(df)} states in {elapsed:.3f}s")

# --- Interactive pyinstrument profiling ---
print("\n--- Pyinstrument Profiling ---")
from pyinstrument import Profiler

profiler = Profiler()
profiler.start()
pool = pp.mutagenize('ACGT' * 25, num_mutations=2, mode='random')
df = pool.generate_library(num_seqs=500)
profiler.stop()
print(profiler.output_text(unicode=True, color=True))

# --- CLI commands for more profiling (run in terminal) ---
print("\n--- CLI Commands (run in terminal) ---")
print("# List available workloads")
print("uv run python poolparty/benchmarks/run_profile.py --list")
print()
print("# Profile with pyinstrument")
print("uv run python poolparty/benchmarks/run_profile.py mutagenize --seq-len 100 --num-seqs 1000")
print()
print("# Profile with memray (generates flamegraph)")
print("uv run python poolparty/benchmarks/run_profile.py mutagenize --memray")
print()
print("# Run benchmark suite")
print("uv run pytest poolparty/benchmarks/ -v --benchmark-disable")

Mutagenize (100bp, 2mut, 1000 seqs): 1000 seqs in 0.288s (0.29ms/seq)
Complex DAG (100bp, 1000 seqs): 1000 seqs in 0.342s (0.34ms/seq)

--- Memory Profiling ---


Mutagenize (200bp, 5000 seqs): current=1.37MB, peak=2.75MB

--- Sequential Mode (Enumerate All States) ---
Sequential mutagenize (20bp, 1mut): 60 states in 0.005s

--- Pyinstrument Profiling ---



  _     ._   __/__   _ _  _  _ _/_   Recorded: 11:56:39  Samples:  374
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.376     CPU time: 0.377
/   _/                      v5.1.2

Profile at /var/folders/00/vxv5z6jj6ln6jd3vlr_6xn8w0000gp/T/ipykernel_45041/2708598256.py:48

0.374 ZMQInteractiveShell.run_ast_nodes  IPython/core/interactiveshell.py:3420
└─ 0.373 <module>  /var/folders/00/vxv5z6jj6ln6jd3vlr_6xn8w0000gp/T/ipykernel_45041/2708598256.py:1
   └─ 0.372 generate_library  <@beartype(poolparty.pool.Pool.generate_library) at 0x11d2ed750>:1
      └─ 0.372 Pool.generate_library  poolparty/pool.py:236
         └─ 0.372 generate_library  <@beartype(poolparty.generate_library.generate_library) at 0x11d314af0>:1
            └─ 0.372 generate_library  poolparty/generate_library.py:10
               └─ 0.370 _compute_one  poolparty/generate_library.py:228
                  ├─ 0.358 compute  <@beartype(poolparty.operation.Operation.compute) at 0x11d2ef370>:1
                  │  └─ 0.357 M